# M2: FM100 group comparisons

`PLANScores.md` M2: `comparisons.compare_fm100_feature` (Mann-Whitney U + effect size, subject-pooled scores, same convention `beh/scripts/features.py`'s `compare_shape_feature` uses) across `comparisons.FEATURES` (TES, PES_RG, PES_BY, VKS_MajRad, VKS_MinRad, VKS_Angle) for the four requested pairs: CTR vs PD, HC vs protan, HC vs deutan, protan vs deutan.

`VKS_Angle` is periodic (folds to `[0, 180)`) -- compared with the same folded-Mann-Whitney approximation `beh/`'s `orientation_deg` already uses (valid as long as a group's angles don't straddle the 0/180 wrap point; see `comparisons.py`'s module docstring).

In [1]:
import sys
sys.path.append('../scripts')

import pandas as pd

import comparisons
import loader

df = loader.load_fm100_raw()

pairs = [
    ('CTR', None, 'PD', None),
    ('CTR', None, 'CVD', 'protan'),
    ('CTR', None, 'CVD', 'deutan'),
    ('CVD', 'protan', 'CVD', 'deutan'),
]
pair_labels = ['CTR vs PD', 'HC vs protan', 'HC vs deutan', 'protan vs deutan']

In [2]:
rows = []
for (g1, sg1, g2, sg2), label in zip(pairs, pair_labels):
    for feature in comparisons.FEATURES:
        result = comparisons.compare_fm100_feature(df, feature, group1=g1, subgroup1=sg1, group2=g2, subgroup2=sg2)
        rows.append({'comparison': label, **result})
results = pd.DataFrame(rows)
results

,comparison,feature,u_val,p_value,rbc,cles,n1,n2
0,CTR vs PD,TES,36.5,0.028537,-0.580460,0.209770,29,6
1,CTR vs PD,PES_RG,31.5,0.016049,-0.637931,0.181034,29,6
2,CTR vs PD,PES_BY,36.0,0.027072,-0.586207,0.206897,29,6
3,CTR vs PD,VKS_MajRad,38.0,0.031124,-0.563218,0.218391,29,6
4,CTR vs PD,VKS_MinRad,36.0,0.024213,-0.586207,0.206897,29,6
5,CTR vs PD,VKS_Angle,85.0,0.949012,-0.022989,0.488506,29,6
6,HC vs protan,TES,6.0,0.000020,-0.954023,0.022989,29,9
7,HC vs protan,PES_RG,0.0,0.000008,-1.000000,0.000000,29,9
8,HC vs protan,PES_BY,59.0,0.014772,-0.547893,0.226054,29,9
9,HC vs protan,VKS_MajRad,8.0,0.000028,-0.938697,0.030651,29,9


## Just the significant rows (p < 0.05, uncorrected -- matching every other group-comparison battery in this project, e.g. `beh/`'s M1)

In [3]:
results[results['p_value'] < 0.05].sort_values('p_value')

,comparison,feature,u_val,p_value,rbc,cles,n1,n2
16,HC vs deutan,VKS_MinRad,7.0,0.000002,-0.939655,0.030172,29,8
15,HC vs deutan,VKS_MajRad,10.0,0.000007,-0.913793,0.043103,29,8
7,HC vs protan,PES_RG,0.0,0.000008,-1.000000,0.000000,29,9
6,HC vs protan,TES,6.0,0.000020,-0.954023,0.022989,29,9
13,HC vs deutan,PES_RG,1.0,0.000024,-0.991379,0.004310,29,8
9,HC vs protan,VKS_MajRad,8.0,0.000028,-0.938697,0.030651,29,9
12,HC vs deutan,TES,8.5,0.000078,-0.926724,0.036638,29,8
10,HC vs protan,VKS_MinRad,17.0,0.000105,-0.869732,0.065134,29,9
14,HC vs deutan,PES_BY,29.0,0.001415,-0.750000,0.125000,29,8
11,HC vs protan,VKS_Angle,215.0,0.003925,0.647510,0.823755,29,9


## Holm-corrected, per pair (M3)

`comparisons.correct_multiple_comparisons` (Holm, the family-wise-error
default), applied once per comparison (6 features = one family), matching
how `ssvepBeh/`'s own correlation battery is corrected. Family-wise error
control is more conservative than the uncorrected view above -- some rows
significant there don't survive here.

In [4]:
corrected = pd.concat([comparisons.correct_multiple_comparisons(results[results['comparison'] == label]) for label in pair_labels])
corrected[corrected['significant']].sort_values('p_corrected')

,comparison,feature,u_val,p_value,rbc,cles,n1,n2,p_corrected,significant
16,HC vs deutan,VKS_MinRad,7.0,0.000002,-0.939655,0.030172,29,8,0.000014,True
15,HC vs deutan,VKS_MajRad,10.0,0.000007,-0.913793,0.043103,29,8,0.000035,True
7,HC vs protan,PES_RG,0.0,0.000008,-1.000000,0.000000,29,9,0.000048,True
13,HC vs deutan,PES_RG,1.0,0.000024,-0.991379,0.004310,29,8,0.000096,True
6,HC vs protan,TES,6.0,0.000020,-0.954023,0.022989,29,9,0.000102,True
9,HC vs protan,VKS_MajRad,8.0,0.000028,-0.938697,0.030651,29,9,0.000112,True
12,HC vs deutan,TES,8.5,0.000078,-0.926724,0.036638,29,8,0.000235,True
10,HC vs protan,VKS_MinRad,17.0,0.000105,-0.869732,0.065134,29,9,0.000314,True
14,HC vs deutan,PES_BY,29.0,0.001415,-0.750000,0.125000,29,8,0.002831,True
11,HC vs protan,VKS_Angle,215.0,0.003925,0.647510,0.823755,29,9,0.007850,True
